In [2]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder

# Categorical Variables

**Categorical variable** adalah variabel yang memiliki nilai berupa kategori terbatas, bukan angka kontinu (contoh: warna, jenis, kondisi).
Model machine learning tidak dapat langsung memproses data bertipe string/object, sehingga perlu dilakukan *encoding* terlebih dahulu.

Ada tiga pendekatan utama:
1. **Drop** — Hapus kolom kategorikal sepenuhnya
2. **Ordinal Encoding** — Ganti setiap kategori dengan bilangan bulat
3. **One-Hot Encoding** — Buat kolom biner baru untuk setiap kategori

In [3]:
X_1 = pd.read_csv("D:/file baru/mendata/Intermediate Machine Learning/train.csv")
X_2 = pd.read_csv("D:/file baru/mendata/Intermediate Machine Learning/test.csv")

## Persiapan Data: Train-Validation Split

Sebelum melatih model, kita perlu:
- Menghapus baris yang tidak memiliki nilai target (`SalePrice`)
- Menghapus kolom yang mengandung nilai kosong (*missing values*)
- Membagi data menjadi **training set** (80%) dan **validation set** (20%)

In [4]:
X_1.dropna(axis=0, subset=['SalePrice'], inplace=True)
y = X_1['SalePrice']
X_1.drop(['SalePrice'], axis=1, inplace=True)

cols_with_missing = [col for col in X_1.columns
                     if X_1[col].isnull().any()]
X_1.drop(cols_with_missing, axis=1, inplace=True)

X_train, X_valid, y_train, y_valid = train_test_split(X_1, y,
                                                      train_size=0.8, test_size=0.2,
                                                      random_state=0)



In [5]:
print(X_1.shape)
print(X_1.columns)

(1460, 61)
Index(['Id', 'MSSubClass', 'MSZoning', 'LotArea', 'Street', 'LotShape',
       'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood',
       'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'OverallQual',
       'OverallCond', 'YearBuilt', 'YearRemodAdd', 'RoofStyle', 'RoofMatl',
       'Exterior1st', 'Exterior2nd', 'ExterQual', 'ExterCond', 'Foundation',
       'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating',
       'HeatingQC', 'CentralAir', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF',
       'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath',
       'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual', 'TotRmsAbvGrd',
       'Functional', 'Fireplaces', 'GarageCars', 'GarageArea', 'PavedDrive',
       'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch',
       'ScreenPorch', 'PoolArea', 'MiscVal', 'MoSold', 'YrSold', 'SaleType',
       'SaleCondition'],
      dtype='str')


## Fungsi Evaluasi: `score_dataset`

Fungsi ini melatih model `RandomForestRegressor` dan mengembalikan nilai **MAE (Mean Absolute Error)** pada validation set.
MAE yang lebih rendah menandakan prediksi yang lebih akurat.

In [6]:
def score_dataset(X_train, X_valid, y_train, y_valid):
    model = RandomForestRegressor(n_estimators=100, random_state=0)
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)
    return mean_absolute_error(y_valid, preds)

## Pendekatan 1: Drop Kolom Kategorikal

Cara paling sederhana: **hapus semua kolom bertipe `object`** sehingga hanya kolom numerik yang tersisa.
Gunakan pendekatan ini jika kolom kategorikal dianggap tidak membawa informasi yang signifikan.
Kelemahannya adalah kita kehilangan potensi informasi dari kolom yang dibuang.

In [7]:
drop_X_train = X_train.select_dtypes(exclude=['object'])
drop_X_valid = X_valid.select_dtypes(exclude=['object'])

print("MAE dari cara 1 adalah: ")
print(score_dataset(drop_X_train, drop_X_valid, y_train, y_valid))


MAE dari cara 1 adalah: 
17952.591404109586


## Identifikasi Kolom Kategorikal

Sebelum melakukan *encoding*, kita perlu memisahkan kolom kategorikal menjadi dua kelompok:
- **`good_label_cols`**: Kolom yang nilainya di validation set sudah tercakup di training set → **aman untuk di-encode**
- **`bad_label_cols`**: Kolom yang mengandung nilai baru di validation set yang tidak ada di training set → **perlu di-drop** agar tidak menyebabkan error saat encoding

In [8]:
# Categorical columns in the training data
object_cols = X_train.select_dtypes(
    include=['object', 'string', 'category']
).columns.tolist()

# Columns that can be safely ordinal encoded
good_label_cols = [
    col for col in object_cols
    if set(X_valid[col].dropna()).issubset(
        set(X_train[col].dropna())
    )
]

# Problematic columns that will be dropped
bad_label_cols = list(
    set(object_cols) - set(good_label_cols)
)

print(
    'Categorical columns that will be ordinal encoded:',
    good_label_cols
)

print(
    '\nCategorical columns that will be dropped:',
    bad_label_cols
)

Categorical columns that will be ordinal encoded: ['MSZoning', 'Street', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'BldgType', 'HouseStyle', 'RoofStyle', 'Exterior1st', 'Exterior2nd', 'ExterQual', 'ExterCond', 'Foundation', 'Heating', 'HeatingQC', 'CentralAir', 'KitchenQual', 'PavedDrive', 'SaleType', 'SaleCondition']

Categorical columns that will be dropped: ['Functional', 'Condition2', 'RoofMatl']


## Pendekatan 2: Ordinal Encoding

**Ordinal Encoding** mengganti setiap nilai kategori dengan angka integer (misalnya: `Never` → 0, `Rarely` → 1, `Frequently` → 2).
Cocok digunakan saat terdapat **urutan yang bermakna** antar kategori.

Langkah-langkah:
1. Hapus kolom yang tidak bisa di-encode (`bad_label_cols`)
2. Terapkan `OrdinalEncoder` pada `good_label_cols`: `fit_transform` pada training set, `transform` pada validation set

In [9]:
# Drop categorical columns that will not be encoded
label_X_train = X_train.drop(bad_label_cols, axis=1).copy()
label_X_valid = X_valid.drop(bad_label_cols, axis=1).copy()

# Apply ordinal encoder 
ordinal_encoder = OrdinalEncoder()
label_X_train[good_label_cols] =ordinal_encoder.fit_transform(X_train[good_label_cols]) # Your code here
label_X_valid[good_label_cols] =ordinal_encoder.transform(X_valid[good_label_cols])  


In [10]:
print("MAE from Approach 2 (Ordinal Encoding):") 
print(score_dataset(label_X_train, label_X_valid, y_train, y_valid))

MAE from Approach 2 (Ordinal Encoding):
17251.037739726027


## Output Ordinal Encoding

Setelah `OrdinalEncoder` diterapkan, setiap nilai kategori diganti dengan **bilangan bulat** (0, 1, 2, ...).
Contoh di bawah menampilkan perbandingan **sebelum** dan **sesudah** encoding pada beberapa kolom pilihan:
- `MSZoning` — zona wilayah
- `Street` — jenis jalan
- `ExterQual` — kualitas eksterior
- `KitchenQual` — kualitas dapur

> **Catatan:** Angka yang dihasilkan **tidak mengandung makna urutan** secara otomatis — urutan ditentukan oleh alphabet default encoder.

In [11]:
# Kolom sampel untuk divisualisasikan
sample_cols = ['MSZoning', 'Street', 'ExterQual', 'KitchenQual']
n_rows = 8

print("=" * 65)
print("SEBELUM Ordinal Encoding (nilai asli kategori)")
print("=" * 65)
print(X_train[sample_cols].head(n_rows).to_string())

print()
print("=" * 65)
print("SESUDAH Ordinal Encoding (diganti integer)")
print("=" * 65)
print(label_X_train[sample_cols].head(n_rows).to_string())

print()
print("=" * 65)
print("MAPPING kategori → angka per kolom")
print("=" * 65)
for col, cats in zip(good_label_cols, ordinal_encoder.categories_):
    if col in sample_cols:
        mapping = {cat: i for i, cat in enumerate(cats)}
        print(f"  {col:15s}: {mapping}")

SEBELUM Ordinal Encoding (nilai asli kategori)
     MSZoning Street ExterQual KitchenQual
618        RL   Pave        Ex          Gd
870        RL   Pave        TA          TA
92         RL   Pave        TA          TA
817        RL   Pave        Gd          Gd
302        RL   Pave        Gd          Gd
1454       FV   Pave        Gd          Gd
40         RL   Pave        TA          TA
959        FV   Pave        Gd          Gd

SESUDAH Ordinal Encoding (diganti integer)
      MSZoning  Street  ExterQual  KitchenQual
618        3.0     1.0        0.0          2.0
870        3.0     1.0        3.0          3.0
92         3.0     1.0        3.0          3.0
817        3.0     1.0        2.0          2.0
302        3.0     1.0        2.0          2.0
1454       1.0     1.0        2.0          2.0
40         3.0     1.0        3.0          3.0
959        1.0     1.0        2.0          2.0

MAPPING kategori → angka per kolom
  MSZoning       : {'C (all)': 0, 'FV': 1, 'RH': 2, 'RL': 3, 'R

## Menghitung Jumlah Nilai Unik per Kolom Kategorikal

Sebelum melakukan One-Hot Encoding, kita perlu mengetahui **kardinalitas** (jumlah nilai unik) setiap kolom kategorikal.
Kolom dengan kardinalitas tinggi akan menghasilkan sangat banyak kolom baru jika di-*one-hot encode*, sehingga perlu dipertimbangkan apakah akan diikutkan atau di-drop.

In [12]:
object_unique = list(map(lambda col: X_train[col].nunique(), object_cols))
d = dict(zip(object_cols, object_unique))

sorted(d.items(), key=lambda x: x[1])

[('Street', 2),
 ('Utilities', 2),
 ('CentralAir', 2),
 ('LandSlope', 3),
 ('PavedDrive', 3),
 ('LotShape', 4),
 ('LandContour', 4),
 ('ExterQual', 4),
 ('KitchenQual', 4),
 ('MSZoning', 5),
 ('LotConfig', 5),
 ('BldgType', 5),
 ('ExterCond', 5),
 ('HeatingQC', 5),
 ('Condition2', 6),
 ('RoofStyle', 6),
 ('Foundation', 6),
 ('Heating', 6),
 ('Functional', 6),
 ('SaleCondition', 6),
 ('RoofMatl', 7),
 ('HouseStyle', 8),
 ('Condition1', 9),
 ('SaleType', 9),
 ('Exterior1st', 15),
 ('Exterior2nd', 16),
 ('Neighborhood', 25)]

## Pemisahan Kolom Berdasarkan Kardinalitas

Untuk One-Hot Encoding, kita hanya akan meng-encode kolom dengan kardinalitas rendah (< 10 nilai unik) agar tidak menambah terlalu banyak dimensi pada dataset.
- **`low_cardinality_cols`**: Kolom dengan < 10 nilai unik → akan di-*one-hot encode*
- **`high_cardinality_cols`**: Kolom dengan ≥ 10 nilai unik → akan di-drop

In [13]:
# Columns that will be one-hot encoded
low_cardinality_cols = [col for col in object_cols if X_train[col].nunique() < 10]

# Columns that will be dropped from the dataset
high_cardinality_cols = list(set(object_cols)-set(low_cardinality_cols))

print('Categorical columns that will be one-hot encoded:', low_cardinality_cols)
print('\nCategorical columns that will be dropped from the dataset:', high_cardinality_cols)

Categorical columns that will be one-hot encoded: ['MSZoning', 'Street', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'ExterQual', 'ExterCond', 'Foundation', 'Heating', 'HeatingQC', 'CentralAir', 'KitchenQual', 'Functional', 'PavedDrive', 'SaleType', 'SaleCondition']

Categorical columns that will be dropped from the dataset: ['Neighborhood', 'Exterior1st', 'Exterior2nd']


## Pendekatan 3: One-Hot Encoding

**One-Hot Encoding** membuat kolom biner baru untuk setiap nilai unik dalam kolom kategorikal.
Misalnya kolom `Color` dengan nilai `['Red', 'Blue', 'Green']` akan menghasilkan tiga kolom baru: `Color_Red`, `Color_Blue`, `Color_Green`.

Cocok digunakan ketika **tidak ada urutan alami** antar kategori (*nominal data*).
Gunakan `handle_unknown='ignore'` dan `sparse_output=False` agar encoder tidak error saat menemui nilai baru di data baru.

In [14]:
# Kardinalitas rendah: sangat cocok untuk One-Hot Encoding
low_cardinality_cols = [col for col in object_cols if X_train[col].nunique() < 10]

# Kardinalitas tinggi: sangat tidak cocok untuk One-Hot Encoding
high_cardinality_cols = list(set(object_cols)-set(low_cardinality_cols))

print('Categorical columns that will be one-hot encoded:', low_cardinality_cols)
print('\nCategorical columns that will be dropped from the dataset:', high_cardinality_cols)

Categorical columns that will be one-hot encoded: ['MSZoning', 'Street', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'ExterQual', 'ExterCond', 'Foundation', 'Heating', 'HeatingQC', 'CentralAir', 'KitchenQual', 'Functional', 'PavedDrive', 'SaleType', 'SaleCondition']

Categorical columns that will be dropped from the dataset: ['Neighborhood', 'Exterior1st', 'Exterior2nd']


# ONE-HOT ENCODING
Implementasi menggunakan kardinalitas rendah.

In [15]:
OH_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
OH_X_train = pd.DataFrame(OH_encoder.fit_transform(X_train[low_cardinality_cols])) # Your code here
OH_X_valid = pd.DataFrame(OH_encoder.transform(X_valid[low_cardinality_cols])) # Your code here

OH_X_train.index = X_train.index
OH_X_valid.index = X_valid.index

num_X_train = X_train.drop(object_cols, axis= 1)
num_X_valid = X_valid.drop(object_cols, axis= 1)

# Add one-hot encoded columns to numerical features
OH_X_train = pd.concat([num_X_train, OH_X_train], axis=1)
OH_X_valid = pd.concat([num_X_valid, OH_X_valid], axis=1)

# Ensure all columns have string type
OH_X_train.columns = OH_X_train.columns.astype(str)
OH_X_valid.columns = OH_X_valid.columns.astype(str)

## Output One-Hot Encoding

Setelah `OneHotEncoder` diterapkan, setiap nilai unik dalam kolom kategorikal menjadi **kolom biner baru** (bernilai 0 atau 1).

Contoh: kolom `Street` dengan 2 nilai unik (`Grvl`, `Pave`) → menghasilkan **2 kolom baru**:
- `Street_Grvl` → 1 jika jalan berbatu kerikil, 0 jika tidak
- `Street_Pave` → 1 jika jalan beraspal, 0 jika tidak

Visualisasi di bawah menampilkan perbandingan **sebelum** dan **sesudah** encoding, serta menampilkan nama kolom baru yang dihasilkan.

In [16]:
# ── Rekonstruksi encoder khusus untuk visualisasi ──────────────────────────
demo_cols = ['Street', 'CentralAir', 'ExterQual']
n_rows = 8

ohe_demo = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe_demo.fit(X_train[demo_cols])

# Nama kolom hasil OHE
ohe_feature_names = ohe_demo.get_feature_names_out(demo_cols)

# Transform
before_df = X_train[demo_cols].head(n_rows).reset_index(drop=True)
after_arr  = ohe_demo.transform(X_train[demo_cols].head(n_rows))
after_df   = pd.DataFrame(after_arr, columns=ohe_feature_names).astype(int)

print("=" * 65)
print("SEBELUM One-Hot Encoding (nilai asli kategori)")
print("=" * 65)
print(before_df.to_string())

print()
print("=" * 65)
print("SESUDAH One-Hot Encoding (kolom biner baru)")
print("=" * 65)
print(after_df.to_string())

print()
print("=" * 65)
print(f"Jumlah kolom SEBELUM OHE (hanya demo_cols): {len(demo_cols)}")
print(f"Jumlah kolom SESUDAH  OHE                 : {len(ohe_feature_names)}")
print()
print("Kolom baru yang dihasilkan:")
for col in ohe_feature_names:
    print(f"  {col}")

print()
print("=" * 65)
print(f"Total kolom dataset SEBELUM OHE: {X_train.shape[1]}")
print(f"Total kolom dataset SESUDAH  OHE (OH_X_train): {OH_X_train.shape[1]}")

SEBELUM One-Hot Encoding (nilai asli kategori)
  Street CentralAir ExterQual
0   Pave          Y        Ex
1   Pave          N        TA
2   Pave          Y        TA
3   Pave          Y        Gd
4   Pave          Y        Gd
5   Pave          Y        Gd
6   Pave          Y        TA
7   Pave          Y        Gd

SESUDAH One-Hot Encoding (kolom biner baru)
   Street_Grvl  Street_Pave  CentralAir_N  CentralAir_Y  ExterQual_Ex  ExterQual_Fa  ExterQual_Gd  ExterQual_TA
0            0            1             0             1             1             0             0             0
1            0            1             1             0             0             0             0             1
2            0            1             0             1             0             0             0             1
3            0            1             0             1             0             0             1             0
4            0            1             0             1             0         

## Menghitung MAE — Pendekatan 3: One-Hot Encoding

Bandingkan hasil MAE ketiga pendekatan:

| Pendekatan | Metode | MAE (approx) |
|---|---|---|
| 1 | Drop kolom kategorikal | ~17,952 |
| 2 | Ordinal Encoding | ~17,251 |
| 3 | One-Hot Encoding | ~17,525 |

> MAE terendah → model terbaik. Ordinal Encoding unggul di dataset ini karena Random Forest tidak terpengaruh signifikan oleh asumsi ordinalitas.

In [17]:
print("MAE from Approach 3 (One-Hot Encoding):") 
print(score_dataset(OH_X_train, OH_X_valid, y_train, y_valid))

MAE from Approach 3 (One-Hot Encoding):


17514.224246575344
